In [1]:
import psycopg2 as pg
import pandas as pd
import pandas.io.sql as psql
import numpy as np
connection = pg.connect("host=localhost dbname=Simulacred user=postgres password=12345678")

In [2]:
bins = pd.IntervalIndex.from_tuples([(0, 30), (31, 60), (61, 90), (91, 120),
                                     (121, 150), (151, 180), (181, 210),
                                     (211, 240), (241, 270), (271, 300),
                                     (301, 330), (331, 360), (361, np.inf)], closed='both')

In [3]:
i = 0

zero = {'ref': [i] * 13, 
    'contrato': ['00000000000'] * 13, 
    'class_atraso_0':  range(30, 400, 30)
   }
zero = pd.DataFrame(zero)
zero['class_atraso_1'] = pd.cut(zero['class_atraso_0'], bins)
zero['class_atraso_0'] = pd.cut(zero['class_atraso_0'], bins)
query = 'SELECT * FROM jpcosta.operacoes where ref = ' + str(i)
mes0 = psql.read_sql(query, connection)[['ref', 'contrato', 'atraso']]
query = 'SELECT * FROM jpcosta.operacoes where ref = ' + str(i + 1)
mes1 = psql.read_sql(query, connection)[['contrato', 'atraso']]
mes0['class_atraso_0'] = pd.cut(mes0['atraso'], bins)
mes1['class_atraso_1'] = pd.cut(mes1['atraso'], bins)

mes0['class_atraso_0'] = pd.cut(mes0['atraso'], bins)
mes1['class_atraso_1'] = pd.cut(mes1['atraso'], bins)

migracao = zero.append(mes0[['ref', 'contrato', 'class_atraso_0']].merge(mes1[['contrato', 'class_atraso_1']], on=('contrato'), how = 'left'))

cross = pd.crosstab([migracao.ref, migracao.class_atraso_0], migracao.class_atraso_1, normalize='index')

len(np.array(cross))



13

In [4]:


#b = psql.read_sql('SELECT min(ref) as n FROM jpcosta.operacoes where atraso > 360', connection).n[0] + 1

n = psql.read_sql('SELECT max(ref) as n FROM jpcosta.operacoes', connection).n[0]

a = []

for i in range(12, n):
    
    zero = {'ref': [i] * 13, 
        'contrato': ['00000000000'] * 13, 
        'class_atraso_0':  range(30, 400, 30)
       }

    zero = pd.DataFrame(zero)

    zero['class_atraso_1'] = pd.cut(zero['class_atraso_0'], bins)
    zero['class_atraso_0'] = pd.cut(zero['class_atraso_0'], bins)

    query = 'SELECT * FROM jpcosta.operacoes where ref = ' + str(i)

    mes0 = psql.read_sql(query, connection)[['ref', 'contrato', 'atraso']]

    query = 'SELECT * FROM jpcosta.operacoes where ref = ' + str(i + 1)

    mes1 = psql.read_sql(query, connection)[['contrato', 'atraso']]

    mes0['class_atraso_0'] = pd.cut(mes0['atraso'], bins)
    mes1['class_atraso_1'] = pd.cut(mes1['atraso'], bins)

    migracao = zero.append(mes0[['ref', 'contrato', 'class_atraso_0']].merge(mes1[['contrato', 'class_atraso_1']], on=('contrato'), how = 'left'))

    l = np.array(pd.crosstab([migracao.ref, migracao.class_atraso_0], migracao.class_atraso_1, normalize='index'))

    a.append(l)
    #matriz = np.concatenate((matriz, np.array(a)), axis=1)
    #m = m.append(a)
c = pd.crosstab([migracao.ref, migracao.class_atraso_0], migracao.class_atraso_1, normalize='index').columns

In [315]:
pd.DataFrame(a[1] - a[0], columns = c).set_index(c)

class_atraso_1,"[0.0, 30.0]","[31.0, 60.0]","[61.0, 90.0]","[91.0, 120.0]","[121.0, 150.0]","[151.0, 180.0]","[181.0, 210.0]","[211.0, 240.0]","[241.0, 270.0]","[271.0, 300.0]","[301.0, 330.0]","[331.0, 360.0]","[361.0, inf]"
class_atraso_1,,,,,,,,,,,,,
"[0.0, 30.0]",0.000160,-0.000160,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"[31.0, 60.0]",-0.133333,-0.016667,0.150000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"[61.0, 90.0]",-0.168831,0.000000,0.051948,0.116883,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"[91.0, 120.0]",-0.277778,0.000000,0.000000,0.055556,0.222222,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"[121.0, 150.0]",-0.133333,0.000000,0.000000,0.000000,-0.133333,0.266667,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"[151.0, 180.0]",0.000000,0.000000,0.000000,0.000000,0.000000,-0.500000,0.5,0.0,0.0,0.0,0.0,0.0,0.0
"[181.0, 210.0]",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"[211.0, 240.0]",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"[241.0, 270.0]",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
